# KPI-Extraktion aus Geschäftsberichten

Dieses Notebook extrahiert Emissions- und Nachhaltigkeitskennzahlen aus PDF-Berichten mithilfe der Heuristiken aus `src/greenwashing_pipeline/kpi_extraction.py`. Die gefundenen Kennzahlen werden als CSV gespeichert und dienen später als Evidenz für das LLM.

## Voraussetzungen
- Installation der Abhängigkeiten aus `requirements.txt` (u. a. `pdfplumber`, `pandas`).
- Die zu analysierenden Finanz- oder Geschäftsberichte liegen als PDF-Datei vor.

In [ ]:
from pathlib import Path
import pandas as pd

import sys
PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from greenwashing_pipeline.document_loader import PDFDocumentLoader
from greenwashing_pipeline.kpi_extraction import KPIExtractor

## Eingaben konfigurieren
Passe `pdf_path`, `output_csv` und optional `max_pages` an deine Testdokumente an.

In [ ]:
pdf_path = Path("Annual Reports/annual-report-2024.pdf.downloadasset.pdf")
output_csv = Path("data/kpis.csv")
max_pages = 5  # für vollständige Verarbeitung auf None setzen

output_csv.parent.mkdir(parents=True, exist_ok=True)
print(f"PDF: {pdf_path}")
print(f"KPI-Datei: {output_csv}")

## PDF laden und vorbereiten

In [ ]:
loader = PDFDocumentLoader(max_pages=max_pages)
sections = loader.load(pdf_path)
print(f"Geladene Abschnitte: {len(sections)}")
if sections:
    print("Erste Textpassage:
", sections[0].text[:500])

## KPIs extrahieren

In [ ]:
kpi_extractor = KPIExtractor()
kpis = kpi_extractor.extract_from_sections(sections)
print(f"Gefundene KPIs: {len(kpis)}")

## Ergebnis speichern

In [ ]:
kpi_df = pd.DataFrame([kpi.to_dict() for kpi in kpis])

kpi_df.to_csv(output_csv, index=False)
kpi_df.head()

Die KPI-CSV dient als Grundlage für den DeepSeek-Abgleich mit den Claims.